# LeetCode 734 — Sentence Similarity bằng Python

**Link đề:** https://leetcode.com/problems/sentence-similarity/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, String

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class Solution:
    def areSentencesSimilar(self, sentence1: list[str], sentence2: list[str], similarPairs: list[list[str]]) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu quan hệ "tương tự" (similar) của đề: **đối xứng**, **phản xạ** (từ giống chính nó), nhưng **không bắc cầu**.
2. Ôn `zip`, `tuple`, `set`, `dict` + `defaultdict(set)`, `all()`.
3. Đi từ cách trực giác (dò cả list cặp cho mỗi vị trí, `O(n·p)`) đến tra cứu **hash set** `O(n + p)`.
4. Trace, chứng minh, phân tích độ phức tạp, test đối chiếu nhiều cách.
5. Thấy được bước nhảy sang bài khó hơn: 737 Sentence Similarity II (có bắc cầu → Union-Find / DFS).

## 1. Đề bài

Một câu được biểu diễn bằng **mảng các từ**, ví dụ `"I am happy with leetcode"` → `["I","am","happy","with","leetcode"]`.

Cho hai câu `sentence1`, `sentence2` và mảng `similarPairs`, trong đó `similarPairs[i] = [x_i, y_i]` nghĩa là hai từ `x_i` và `y_i` **tương tự**.

Hai câu được coi là **tương tự** khi:
1. Có **cùng số từ**, và
2. Với mọi vị trí `i`, `sentence1[i]` và `sentence2[i]` tương tự.

Quy ước về quan hệ tương tự:
- Một từ **luôn tương tự chính nó** (dù không có trong `similarPairs`).
- Quan hệ **đối xứng**: có `[a, b]` thì `a ~ b` và `b ~ a`.
- Quan hệ **không bắc cầu**: `a ~ b` và `b ~ c` **không** suy ra `a ~ c`.

Trả về `True` nếu hai câu tương tự, ngược lại `False`.

| sentence1 | sentence2 | similarPairs | Kết quả |
|---|---|---|:---:|
| `["great","acting","skills"]` | `["fine","drama","talent"]` | `[["great","fine"],["drama","acting"],["skills","talent"]]` | `True` |
| `["great"]` | `["great"]` | `[]` | `True` (từ giống chính nó) |
| `["great"]` | `["doubleplus","good"]` | `[["great","doubleplus"]]` | `False` (khác độ dài) |

**Constraints (theo hiểu biết):** `1 <= len(sentence1), len(sentence2) <= 1000`; mỗi từ dài `1..20`, gồm chữ cái tiếng Anh (hoa/thường, **phân biệt hoa thường**); `0 <= len(similarPairs) <= 1000`; mỗi cặp có đúng 2 từ; các cặp đôi một khác nhau.

In [1]:
sentence1 = ["great", "acting", "skills"]
sentence2 = ["fine", "drama", "talent"]
similarPairs = [["great", "fine"], ["drama", "acting"], ["skills", "talent"]]
print("Độ dài:", len(sentence1), len(sentence2))
for i in range(len(sentence1)):
    print(i, sentence1[i], "<->", sentence2[i])

Độ dài: 3 3
0 great <-> fine
1 acting <-> drama
2 skills <-> talent


Chú ý vị trí 1: cặp cho là `["drama","acting"]` nhưng câu cần `acting ~ drama` → nhờ tính **đối xứng** vẫn hợp lệ. Đây chính là chỗ người mới hay quên.

## 2. Nền tảng Python

### 2.1. `zip` — đi song song hai list

`zip(a, b)` ghép phần tử cùng vị trí thành cặp `(a[i], b[i])`. Nếu hai list dài khác nhau, `zip` **dừng ở list ngắn hơn** — không báo lỗi! Vì vậy phải tự kiểm tra độ dài trước.

In [2]:
for w1, w2 in zip(["a", "b", "c"], ["x", "y", "z"]):
    print(w1, w2)
print(list(zip(["great"], ["doubleplus", "good"])))   # chỉ 1 cặp, "good" bị bỏ qua lặng lẽ

a x
b y
c z
[('great', 'doubleplus')]


### 2.2. `tuple` — cặp không đổi được, dùng làm key

`list` **không** thể cho vào `set` hay làm key `dict` (vì list thay đổi được → không có hash ổn định). `tuple` thì được. Ta đổi mỗi cặp `["great","fine"]` thành `("great","fine")`.

In [3]:
pair_list = ["great", "fine"]
pair_tuple = tuple(pair_list)
print(pair_tuple, type(pair_tuple).__name__)
s = set()
s.add(pair_tuple)
print(("great", "fine") in s, ("fine", "great") in s)   # True False -> thứ tự trong tuple có ý nghĩa
try:
    s.add(pair_list)
except TypeError as e:
    print("Không thêm list vào set được:", e)

('great', 'fine') tuple
True False
Không thêm list vào set được: unhashable type: 'list'


### 2.3. `set` — kiểm tra "có trong tập không" nhanh

`x in my_set` trung bình `O(1)` (hash table), còn `x in my_list` là `O(len)` vì phải dò từng phần tử.

In [4]:
import time

data_list = [("w%d" % i, "v%d" % i) for i in range(1000)]
data_set = set(data_list)
probe = ("w999", "v999")

t0 = time.perf_counter()
for _ in range(2000):
    probe in data_list
t_list = time.perf_counter() - t0

t0 = time.perf_counter()
for _ in range(2000):
    probe in data_set
t_set = time.perf_counter() - t0
print(f"list: {t_list:.4f}s | set: {t_set:.5f}s  (set nhanh hơn rất nhiều)")

list: 0.0267s | set: 0.00023s  (set nhanh hơn rất nhiều)


### 2.4. `defaultdict(set)` — dict tự tạo giá trị mặc định

`defaultdict(set)` giống `dict`, nhưng khi truy cập key chưa có, nó tự tạo `set()` rỗng. Tiện để xây "danh sách kề": từ → tập các từ tương tự.

In [5]:
from collections import defaultdict

sim = defaultdict(set)
sim["great"].add("fine")       # không cần kiểm tra "great" đã có chưa
sim["fine"].add("great")
sim["great"].add("good")
print(dict(sim))
print("good" in sim["great"], "bad" in sim["nothing"])   # key mới -> set rỗng

{'great': {'good', 'fine'}, 'fine': {'great'}}
True False


### 2.5. `all()` với generator

`all(cond(x) for x in xs)` = "mọi phần tử thỏa điều kiện?". Dừng ngay ở phần tử sai đầu tiên.

In [6]:
print(all(x > 0 for x in [3, 5, 1]))
print(all(x > 0 for x in [3, -5, 1]))
print(all(x > 0 for x in []))          # True: không có phần tử nào vi phạm

True
False
True


## 3. Cách 1 — Trực giác: dò cả list cặp cho từng vị trí

1. Khác độ dài → `False`.
2. Với mỗi vị trí `i`, lấy `w1 = sentence1[i]`, `w2 = sentence2[i]`:
   - Nếu `w1 == w2` → ổn.
   - Nếu không, duyệt toàn bộ `similarPairs` tìm cặp `[w1, w2]` **hoặc** `[w2, w1]`. Không thấy → `False`.
3. Qua hết → `True`.

In [7]:
def similar_bruteforce(sentence1, sentence2, similarPairs):
    if len(sentence1) != len(sentence2):
        return False
    for w1, w2 in zip(sentence1, sentence2):
        if w1 == w2:
            continue
        found = False
        for x, y in similarPairs:
            if (x == w1 and y == w2) or (x == w2 and y == w1):
                found = True
                break
        if not found:
            return False
    return True

print(similar_bruteforce(["great", "acting", "skills"], ["fine", "drama", "talent"],
                         [["great", "fine"], ["drama", "acting"], ["skills", "talent"]]))
print(similar_bruteforce(["great"], ["great"], []))
print(similar_bruteforce(["great"], ["doubleplus", "good"], [["great", "doubleplus"]]))

True
True
False


### Nút thắt của Cách 1

Gọi `n` = số từ mỗi câu, `p` = số cặp. Mỗi vị trí có thể phải dò hết `p` cặp → **`O(n · p)`** phép so sánh chuỗi. Với `n = p = 1000` là `10^6` lần so sánh — vẫn qua được, nhưng ta đang **lặp lại cùng một việc**: dò lại list `similarPairs` từ đầu cho mỗi vị trí.

**Insight:** câu hỏi ở mỗi vị trí chỉ là "*cặp `(w1, w2)` có thuộc tập các cặp tương tự không?*" → đây đúng là việc của **hash set**. Ta tiền xử lý `similarPairs` **một lần** thành tập, sau đó mỗi câu hỏi chỉ tốn `O(1)` trung bình.

Để xử lý đối xứng, khi xây tập ta thêm **cả hai chiều** `(x, y)` và `(y, x)` — sau đó chỉ cần tra một chiều.

In [8]:
pairs = [["great", "fine"], ["drama", "acting"]]
pair_set = set()
for x, y in pairs:
    pair_set.add((x, y))
    pair_set.add((y, x))
print(pair_set)
print(("acting", "drama") in pair_set)   # True nhờ thêm chiều ngược

{('fine', 'great'), ('drama', 'acting'), ('great', 'fine'), ('acting', 'drama')}
True


## 4. Cách 2 — Set các cặp hai chiều (lời giải chính)

Ghép từng đoạn:

**Đoạn A — loại sớm khác độ dài:**
```python
if len(sentence1) != len(sentence2):
    return False
```

**Đoạn B — tiền xử lý:** `pair_set` chứa `(x, y)` và `(y, x)` cho mọi cặp.

**Đoạn C — kiểm từng vị trí:** vị trí `i` hợp lệ khi `w1 == w2` **hoặc** `(w1, w2) in pair_set`.

In [9]:
def similar_pair_set(sentence1, sentence2, similarPairs):
    if len(sentence1) != len(sentence2):            # Đoạn A
        return False
    pair_set = set()                                 # Đoạn B
    for x, y in similarPairs:
        pair_set.add((x, y))
        pair_set.add((y, x))
    for w1, w2 in zip(sentence1, sentence2):         # Đoạn C
        if w1 != w2 and (w1, w2) not in pair_set:
            return False
    return True

print(similar_pair_set(["great", "acting", "skills"], ["fine", "drama", "talent"],
                       [["great", "fine"], ["drama", "acting"], ["skills", "talent"]]))
print(similar_pair_set(["great"], ["great"], []))
print(similar_pair_set(["great"], ["doubleplus", "good"], [["great", "doubleplus"]]))

True
True
False


Điều kiện trong Đoạn C viết theo kiểu **phủ định** (luật De Morgan):

- "vị trí hợp lệ" = `w1 == w2 or (w1, w2) in pair_set`
- "vị trí **không** hợp lệ" = `w1 != w2 and (w1, w2) not in pair_set`

Gặp vị trí không hợp lệ → trả `False` ngay (early exit).

## 5. Trace từng bước

Ví dụ: `sentence1 = ["great","acting","skills"]`, `sentence2 = ["fine","drama","talent"]`, cặp như ví dụ 1.

`pair_set` sau Đoạn B có 6 phần tử: `(great,fine), (fine,great), (drama,acting), (acting,drama), (skills,talent), (talent,skills)`.

| i | w1 | w2 | w1 == w2? | (w1, w2) in pair_set? | Quyết định |
|---:|---|---|:---:|:---:|---|
| 0 | great | fine | ❌ | ✅ | tiếp |
| 1 | acting | drama | ❌ | ✅ (nhờ chiều ngược) | tiếp |
| 2 | skills | talent | ❌ | ✅ | tiếp |
| — | | | | | hết → `True` |

In [10]:
def trace_similar(sentence1, sentence2, similarPairs):
    print(f"len1 = {len(sentence1)}, len2 = {len(sentence2)}")
    if len(sentence1) != len(sentence2):
        print("Khác độ dài -> False")
        return False
    pair_set = set()
    for x, y in similarPairs:
        pair_set.add((x, y))
        pair_set.add((y, x))
    print("pair_set =", sorted(pair_set))
    for i, (w1, w2) in enumerate(zip(sentence1, sentence2)):
        same = w1 == w2
        in_set = (w1, w2) in pair_set
        verdict = "tiếp" if (same or in_set) else "KHÔNG tương tự -> False"
        print(f"i={i}: {w1!r:>10} vs {w2!r:<10} | bằng nhau={same!s:<5} | trong set={in_set!s:<5} | {verdict}")
        if not (same or in_set):
            return False
    print("Mọi vị trí đều ổn -> True")
    return True

trace_similar(["great", "acting", "skills"], ["fine", "drama", "talent"],
              [["great", "fine"], ["drama", "acting"], ["skills", "talent"]])

len1 = 3, len2 = 3
pair_set = [('acting', 'drama'), ('drama', 'acting'), ('fine', 'great'), ('great', 'fine'), ('skills', 'talent'), ('talent', 'skills')]
i=0:    'great' vs 'fine'     | bằng nhau=False | trong set=True  | tiếp
i=1:   'acting' vs 'drama'    | bằng nhau=False | trong set=True  | tiếp
i=2:   'skills' vs 'talent'   | bằng nhau=False | trong set=True  | tiếp
Mọi vị trí đều ổn -> True


True

### Trace một trường hợp `False` vì **không bắc cầu**

`a ~ b`, `b ~ c` nhưng câu cần `a ~ c`. `(a, c)` không có trong set → `False`. Đây là điểm khác biệt cốt lõi với bài 737.

Trong f-string, `{w1!r}` in theo dạng `repr` (có dấu nháy), `{same!s:<5}` đổi sang chuỗi rồi căn trái 5 ký tự.

In [11]:
trace_similar(["a", "x"], ["c", "x"], [["a", "b"], ["b", "c"]])

len1 = 2, len2 = 2
pair_set = [('a', 'b'), ('b', 'a'), ('b', 'c'), ('c', 'b')]
i=0:        'a' vs 'c'        | bằng nhau=False | trong set=False | KHÔNG tương tự -> False


False

## 6. Cách 3 — `dict` từ → tập từ tương tự (danh sách kề)

Thay vì lưu cặp, lưu **mỗi từ → tập các từ tương tự với nó**. Cách này tự nhiên hơn khi sau này cần mở rộng thành **đồ thị** (bài 737). Kết hợp `all()` cho gọn.

In [12]:
from collections import defaultdict

def similar_adjacency(sentence1, sentence2, similarPairs):
    if len(sentence1) != len(sentence2):
        return False
    sim = defaultdict(set)
    for x, y in similarPairs:
        sim[x].add(y)
        sim[y].add(x)
    return all(w1 == w2 or w2 in sim[w1] for w1, w2 in zip(sentence1, sentence2))

print(similar_adjacency(["great", "acting", "skills"], ["fine", "drama", "talent"],
                        [["great", "fine"], ["drama", "acting"], ["skills", "talent"]]))
print(similar_adjacency(["a", "x"], ["c", "x"], [["a", "b"], ["b", "c"]]))

True
False


Lưu ý nhỏ: `sim[w1]` với `defaultdict` sẽ **tạo thêm** key `w1` nếu chưa có (một set rỗng). Không ảnh hưởng kết quả, nhưng nếu muốn tránh "phình" dict, dùng `sim.get(w1, ())`.

### Cách 3b — chuẩn hóa cặp bằng `frozenset`

`frozenset({x, y})` là tập **không có thứ tự** và **hashable** → `frozenset({"a","b"}) == frozenset({"b","a"})`. Chỉ cần lưu một phần tử cho mỗi cặp.

In [13]:
print(frozenset(["drama", "acting"]) == frozenset(["acting", "drama"]))

def similar_frozenset(sentence1, sentence2, similarPairs):
    if len(sentence1) != len(sentence2):
        return False
    pairs = {frozenset(p) for p in similarPairs}
    return all(w1 == w2 or frozenset((w1, w2)) in pairs
               for w1, w2 in zip(sentence1, sentence2))

print(similar_frozenset(["great", "acting", "skills"], ["fine", "drama", "talent"],
                        [["great", "fine"], ["drama", "acting"], ["skills", "talent"]]))

True
True


`{frozenset(p) for p in similarPairs}` là **set comprehension** — giống list comprehension nhưng dùng ngoặc nhọn và tạo `set`.

## 7. Code nộp LeetCode

### Bản chính (set các cặp hai chiều)

In [14]:
class Solution:
    def areSentencesSimilar(self, sentence1: list[str], sentence2: list[str], similarPairs: list[list[str]]) -> bool:
        if len(sentence1) != len(sentence2):
            return False
        pair_set = set()
        for x, y in similarPairs:
            pair_set.add((x, y))
            pair_set.add((y, x))
        for w1, w2 in zip(sentence1, sentence2):
            if w1 != w2 and (w1, w2) not in pair_set:
                return False
        return True


sol = Solution()
print(sol.areSentencesSimilar(["great", "acting", "skills"], ["fine", "drama", "talent"],
                              [["great", "fine"], ["drama", "acting"], ["skills", "talent"]]))
print(sol.areSentencesSimilar(["great"], ["great"], []))
print(sol.areSentencesSimilar(["great"], ["doubleplus", "good"], [["great", "doubleplus"]]))

True
True
False


### Các bản nộp khác

Trên LeetCode đổi tên class thành `Solution`. (Ở một số phiên bản cũ của đề, tham số tên là `words1, words2, pairs` — chỉ cần giữ đúng **thứ tự** tham số theo skeleton bạn thấy.)

In [15]:
class SolutionAdjacency:
    def areSentencesSimilar(self, sentence1: list[str], sentence2: list[str], similarPairs: list[list[str]]) -> bool:
        if len(sentence1) != len(sentence2):
            return False
        sim = defaultdict(set)
        for x, y in similarPairs:
            sim[x].add(y)
            sim[y].add(x)
        return all(a == b or b in sim[a] for a, b in zip(sentence1, sentence2))


class SolutionBruteForce:
    def areSentencesSimilar(self, sentence1: list[str], sentence2: list[str], similarPairs: list[list[str]]) -> bool:
        return similar_bruteforce(sentence1, sentence2, similarPairs)


print(SolutionAdjacency().areSentencesSimilar(["a"], ["b"], [["b", "a"]]))

True


## 8. Vì sao đúng?

**Tập cặp đúng nghĩa.** Sau Đoạn B: `(u, v) ∈ pair_set` ⟺ có cặp `[u, v]` **hoặc** `[v, u]` trong `similarPairs`. Vì ta thêm cả hai chiều cho mỗi cặp, và không thêm gì khác.

Vậy với mỗi vị trí, điều kiện `w1 == w2 or (w1, w2) in pair_set` **trùng khớp chính xác** định nghĩa "hai từ tương tự" của đề:
- phản xạ ↔ `w1 == w2`;
- có cặp trực tiếp, theo chiều nào cũng được ↔ `(w1, w2) in pair_set`;
- **không** thêm gì vì bắc cầu ↔ ta không bao giờ suy luận ra cặp mới.

**Vòng lặp.** Invariant: *trước vị trí `i`, mọi vị trí `0..i-1` đều tương tự.* Gặp vị trí sai → trả `False` (đúng vì câu bị vi phạm ở một vị trí). Qua hết → mọi vị trí đều ổn → `True`.

**Độ dài.** Phải kiểm riêng vì `zip` âm thầm cắt theo list ngắn hơn.

## 9. Độ phức tạp

Gọi `n = len(sentence1)`, `p = len(similarPairs)`, `L ≤ 20` là độ dài tối đa một từ (băm/so sánh chuỗi tốn `O(L)`).

| Cách | Time | Space phụ |
|---|---|---|
| 1. Dò list cặp | `O(n · p · L)` | `O(1)` |
| 2. Set cặp hai chiều | `O((n + p) · L)` trung bình | `O(p · L)` |
| 3. Dict kề / frozenset | `O((n + p) · L)` trung bình | `O(p · L)` |

Vì `L` bị chặn bởi 20 (hằng số), thường viết gọn: Cách 1 `O(n·p)`, Cách 2/3 `O(n + p)`. Chữ "trung bình" vì tra cứu hash là `O(1)` **trung bình**, không đảm bảo tuyệt đối trong trường hợp xấu.

Một tối ưu nhỏ: nếu `len` khác nhau ta trả `False` **trước** khi xây set → không tốn `O(p)` vô ích.

## 10. Bộ test

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}")

sol = Solution()
P1 = [["great", "fine"], ["drama", "acting"], ["skills", "talent"]]
check("đề 1", sol.areSentencesSimilar(["great", "acting", "skills"], ["fine", "drama", "talent"], P1), True)
check("đề 2 phản xạ", sol.areSentencesSimilar(["great"], ["great"], []), True)
check("đề 3 khác độ dài", sol.areSentencesSimilar(["great"], ["doubleplus", "good"], [["great", "doubleplus"]]), False)
check("đối xứng", sol.areSentencesSimilar(["fine"], ["great"], [["great", "fine"]]), True)
check("không bắc cầu", sol.areSentencesSimilar(["a"], ["c"], [["a", "b"], ["b", "c"]]), False)
check("phân biệt hoa thường", sol.areSentencesSimilar(["Great"], ["great"], []), False)
check("không có cặp", sol.areSentencesSimilar(["a", "b"], ["a", "c"], []), False)
check("vị trí lệch", sol.areSentencesSimilar(["a", "b"], ["b", "a"], [["a", "c"]]), False)
check("cặp tồn tại nhưng sai vị trí", sol.areSentencesSimilar(["great", "x"], ["x", "fine"], [["great", "fine"]]), False)
check("một từ có nhiều bạn", sol.areSentencesSimilar(["good", "good"], ["fine", "nice"], [["good", "fine"], ["nice", "good"]]), True)

PASS  đề 1
PASS  đề 2 phản xạ
PASS  đề 3 khác độ dài
PASS  đối xứng
PASS  không bắc cầu
PASS  phân biệt hoa thường
PASS  không có cặp
PASS  vị trí lệch
PASS  cặp tồn tại nhưng sai vị trí
PASS  một từ có nhiều bạn


In [17]:
import random

rng = random.Random(734)
vocab = ["a", "b", "c", "d", "e"]
impls = [similar_bruteforce, similar_pair_set, similar_adjacency, similar_frozenset,
         Solution().areSentencesSimilar, SolutionAdjacency().areSentencesSimilar]
true_count = 0
for _ in range(3000):
    n1 = rng.randint(1, 4)
    n2 = n1 if rng.random() < 0.85 else rng.randint(1, 4)
    s1 = [rng.choice(vocab) for _ in range(n1)]
    s2 = [rng.choice(vocab) for _ in range(n2)]
    all_pairs = [[x, y] for x in vocab for y in vocab if x < y]
    pairs = rng.sample(all_pairs, rng.randint(0, 6))
    pairs = [p if rng.random() < 0.5 else p[::-1] for p in pairs]
    outs = {f(s1, s2, pairs) for f in impls}
    assert len(outs) == 1, (s1, s2, pairs, outs)
    true_count += outs.pop()
print("3000 bộ ngẫu nhiên: 6 cài đặt khớp nhau. Số bộ cho True:", true_count)

3000 bộ ngẫu nhiên: 6 cài đặt khớp nhau. Số bộ cho True: 538


`p[::-1]` đảo cặp `[x, y]` thành `[y, x]` — để test chắc chắn có cả hai chiều trong dữ liệu ngẫu nhiên.

## 11. Lỗi người mới hay gặp

### Lỗi 1: Quên kiểm tra độ dài (tin vào `zip`)

In [18]:
def wrong_no_length(sentence1, sentence2, similarPairs):
    pair_set = {(x, y) for x, y in similarPairs} | {(y, x) for x, y in similarPairs}
    return all(a == b or (a, b) in pair_set for a, b in zip(sentence1, sentence2))

print("sai :", wrong_no_length(["great"], ["great", "extra"], []), "| đúng:", similar_pair_set(["great"], ["great", "extra"], []))

sai : True | đúng: False


(Toán tử `|` giữa hai set là **hợp** — union.)

### Lỗi 2: Chỉ lưu một chiều của cặp

In [19]:
def wrong_one_direction(sentence1, sentence2, similarPairs):
    if len(sentence1) != len(sentence2):
        return False
    pair_set = {(x, y) for x, y in similarPairs}          # thiếu chiều ngược
    return all(a == b or (a, b) in pair_set for a, b in zip(sentence1, sentence2))

args = (["great", "acting", "skills"], ["fine", "drama", "talent"], P1)
print("sai :", wrong_one_direction(*args), "| đúng:", similar_pair_set(*args))

sai : False | đúng: True


(`f(*args)` **giải nén** tuple `args` thành các tham số riêng lẻ.)

### Lỗi 3: Quên trường hợp hai từ giống nhau

Nếu chỉ tra `pair_set`, câu `["great"]` vs `["great"]` với `similarPairs = []` cho `False` — sai, vì từ luôn tương tự chính nó.

In [20]:
def wrong_no_reflexive(sentence1, sentence2, similarPairs):
    if len(sentence1) != len(sentence2):
        return False
    pair_set = {(x, y) for x, y in similarPairs} | {(y, x) for x, y in similarPairs}
    return all((a, b) in pair_set for a, b in zip(sentence1, sentence2))

print("sai :", wrong_no_reflexive(["great"], ["great"], []), "| đúng:", similar_pair_set(["great"], ["great"], []))

sai : False | đúng: True


### Lỗi 4: Gom từ vào **một set chung** thay vì lưu cặp

Ví dụ `words = {"great","fine","drama","acting"}` rồi kiểm `w1 in words and w2 in words`. Sai vì mất thông tin "**ai đi với ai**": `great` và `drama` đều có trong tập nhưng không tương tự nhau.

In [21]:
def wrong_single_set(sentence1, sentence2, similarPairs):
    if len(sentence1) != len(sentence2):
        return False
    words = {w for p in similarPairs for w in p}
    return all(a == b or (a in words and b in words) for a, b in zip(sentence1, sentence2))

args = (["great"], ["drama"], P1)
print("sai :", wrong_single_set(*args), "| đúng:", similar_pair_set(*args))

sai : True | đúng: False


### Lỗi 5: Tự ý thêm tính bắc cầu (giải nhầm sang bài 737)

Dùng Union-Find/DFS cho bài này sẽ trả `True` cho ví dụ `a~b, b~c` ⇒ `a` vs `c` — **sai** với đề 734. Đọc kỹ: đề nói rõ quan hệ **không** bắc cầu.

## 12. Mẹo nhận dạng pattern

- "Kiểm tra một **cặp** có thuộc danh sách cho trước không", hỏi nhiều lần → tiền xử lý thành **set các tuple** (hoặc dict kề).
- Quan hệ **đối xứng** → thêm cả hai chiều khi xây (hoặc dùng `frozenset`).
- Quan hệ **bắc cầu** (bạn của bạn cũng là bạn, tương đương, cùng nhóm) → đồ thị: **Union-Find** hoặc **DFS/BFS** tìm thành phần liên thông.
- So sánh hai dãy theo vị trí → `zip` + **kiểm độ dài trước**.
- Bài cùng họ: 737 Sentence Similarity II, 1813 Sentence Similarity III, 290 Word Pattern, 205 Isomorphic Strings, 839 Similar String Groups.

## 13. Biến thể

### 13.1. LeetCode 737 — Sentence Similarity II (có bắc cầu)

Giờ `a ~ b`, `b ~ c` ⇒ `a ~ c`. Mỗi "nhóm từ tương tự" là một **thành phần liên thông** của đồ thị. Dùng **Union-Find** (Disjoint Set Union):
- `find(x)`: tìm "đại diện" của nhóm chứa `x` (đi theo `parent` tới gốc).
- `union(x, y)`: gộp hai nhóm bằng cách nối gốc này vào gốc kia.
- Hai từ tương tự ⟺ cùng đại diện.

In [22]:
def similar_transitive(sentence1, sentence2, similarPairs):
    if len(sentence1) != len(sentence2):
        return False
    parent = {}

    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]      # nén đường đi (path halving)
            x = parent[x]
        return x

    def union(x, y):
        parent[find(x)] = find(y)

    for x, y in similarPairs:
        union(x, y)
    return all(a == b or find(a) == find(b) for a, b in zip(sentence1, sentence2))

pairs = [["great", "good"], ["fine", "good"], ["drama", "acting"], ["skills", "talent"]]
print(similar_transitive(["great", "acting", "skills"], ["fine", "drama", "talent"], pairs))   # True (great~good~fine)
print(similar_pair_set(["great", "acting", "skills"], ["fine", "drama", "talent"], pairs))     # False ở bài 734

True
False


`parent.setdefault(x, x)`: nếu `x` chưa là key thì gán `parent[x] = x` (mỗi từ ban đầu là gốc của chính nó), rồi trả giá trị.

### 13.2. Đếm số vị trí **không** tương tự

Thay vì `True/False`, trả về danh sách vị trí lệch — hữu ích để báo lỗi chi tiết.

In [23]:
def mismatch_positions(sentence1, sentence2, similarPairs):
    pair_set = {(x, y) for x, y in similarPairs} | {(y, x) for x, y in similarPairs}
    return [i for i, (a, b) in enumerate(zip(sentence1, sentence2))
            if a != b and (a, b) not in pair_set]

print(mismatch_positions(["I", "love", "leetcode"], ["I", "like", "coding"], [["love", "like"]]))   # [2]

[2]


## 14. Bài tập

### Bài 1 — Đoán output (không chạy)
Với `similarPairs = [["happy","glad"],["glad","joyful"]]`, Cách 2 trả gì cho:
- (a) `["happy"]` vs `["joyful"]`
- (b) `["glad", "x"]` vs `["happy", "x"]`
- (c) `["Happy"]` vs `["glad"]`

### Bài 2 — Xây set hai chiều
Viết `build_pair_set(pairs)` trả set chứa `(x, y)` và `(y, x)` cho mọi cặp.

In [24]:
def build_pair_set(pairs: list[list[str]]) -> set[tuple[str, str]]:
    # TODO
    pass

# assert build_pair_set([["a", "b"]]) == {("a", "b"), ("b", "a")}
# assert build_pair_set([]) == set()

### Bài 3 — Điền chỗ trống lời giải chính

In [25]:
def exercise_are_similar(sentence1, sentence2, similarPairs) -> bool:
    # TODO 1: khác độ dài -> False
    pair_set = set()
    # TODO 2: thêm cả hai chiều cho mỗi cặp
    for w1, w2 in zip(sentence1, sentence2):
        # TODO 3: nếu w1 khác w2 và cặp không có trong set -> False
        pass
    return True

# assert exercise_are_similar(["great","acting","skills"], ["fine","drama","talent"],
#                             [["great","fine"],["drama","acting"],["skills","talent"]]) == True
# assert exercise_are_similar(["great"], ["doubleplus","good"], [["great","doubleplus"]]) == False

### Bài 4 — Không phân biệt hoa thường
Viết biến thể coi `"Great"` và `"great"` là cùng một từ (cả trong câu lẫn trong `similarPairs`).

In [26]:
def similar_ignore_case(sentence1, sentence2, similarPairs) -> bool:
    # TODO
    pass

# assert similar_ignore_case(["Great"], ["great"], []) == True
# assert similar_ignore_case(["GREAT"], ["Fine"], [["great", "fine"]]) == True
# assert similar_ignore_case(["a"], ["c"], [["A", "b"], ["b", "C"]]) == False

### Bài 5 — Có bắc cầu nhưng dùng DFS (không Union-Find)
Giải lại bài 737 bằng cách xây đồ thị kề rồi DFS/BFS từ `w1` xem có tới được `w2` không.

In [27]:
def similar_transitive_dfs(sentence1, sentence2, similarPairs) -> bool:
    # TODO
    pass

# pairs = [["great","good"],["fine","good"],["drama","acting"],["skills","talent"]]
# assert similar_transitive_dfs(["great","acting","skills"], ["fine","drama","talent"], pairs) == True
# assert similar_transitive_dfs(["great"], ["drama"], pairs) == False

### Bài 6 — Mini challenge: LeetCode 1813 Sentence Similarity III (rút gọn)
Hai câu (chuỗi, từ cách nhau bởi 1 dấu cách) tương tự nếu chèn **một** đoạn câu (có thể rỗng) vào **một** chỗ trong câu ngắn hơn thì được câu dài hơn. Ví dụ `"My name is Haley"` và `"My Haley"` → `True` (chèn `"name is"`).

In [28]:
def similar_iii(sentence1: str, sentence2: str) -> bool:
    # TODO
    pass

# assert similar_iii("My name is Haley", "My Haley") == True
# assert similar_iii("of", "A lot of words") == False
# assert similar_iii("Eating right now", "Eating") == True

## 15. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | Không bắc cầu, có đối xứng, phân biệt hoa thường | Kiểm từng cặp vị trí | (a) cần `(happy, joyful)`? (b) `(glad, happy)`? (c) `"Happy" != "happy"` |
| 2 | Mỗi cặp góp 2 tuple | Vòng `for x, y in pairs` | `s.add((x, y)); s.add((y, x))` |
| 3 | Như Cách 2 | Ba TODO = ba đoạn A, B, C | `if w1 != w2 and (w1, w2) not in pair_set: return False` |
| 4 | Chuẩn hóa mọi từ trước khi so | `.lower()` | `s1 = [w.lower() for w in sentence1]`, tương tự cho cặp |
| 5 | Đồ thị vô hướng | `defaultdict(set)` + BFS có `seen` | `bfs(a, b)`: từ `a`, lan qua `graph[u]`, gặp `b` → True |
| 6 | Tiền tố chung + hậu tố chung | Hai con trỏ từ đầu và từ cuối | `i` tăng khi từ đầu khớp, `j` tăng khi từ cuối khớp; `i + j >= len(câu ngắn)` |

## 16. Đáp án tham khảo

**Bài 1:** (a) `False` — `happy~glad`, `glad~joyful` nhưng không bắc cầu. (b) `True` — `glad~happy` nhờ đối xứng, `x == x`. (c) `False` — phân biệt hoa thường, `"Happy"` không có cặp nào.

In [29]:
P = [["happy", "glad"], ["glad", "joyful"]]
print("Bài 1:", similar_pair_set(["happy"], ["joyful"], P),
      similar_pair_set(["glad", "x"], ["happy", "x"], P),
      similar_pair_set(["Happy"], ["glad"], P))


def answer_build_pair_set(pairs):
    s = set()
    for x, y in pairs:
        s.add((x, y))
        s.add((y, x))
    return s


def answer_are_similar(sentence1, sentence2, similarPairs):
    if len(sentence1) != len(sentence2):
        return False
    pair_set = answer_build_pair_set(similarPairs)
    for w1, w2 in zip(sentence1, sentence2):
        if w1 != w2 and (w1, w2) not in pair_set:
            return False
    return True


def answer_ignore_case(sentence1, sentence2, similarPairs):
    s1 = [w.lower() for w in sentence1]
    s2 = [w.lower() for w in sentence2]
    pairs = [[x.lower(), y.lower()] for x, y in similarPairs]
    return answer_are_similar(s1, s2, pairs)


def answer_transitive_dfs(sentence1, sentence2, similarPairs):
    from collections import deque
    if len(sentence1) != len(sentence2):
        return False
    graph = defaultdict(set)
    for x, y in similarPairs:
        graph[x].add(y)
        graph[y].add(x)

    def reachable(a, b):
        seen = {a}
        q = deque([a])
        while q:
            u = q.popleft()
            if u == b:
                return True
            for v in graph[u]:
                if v not in seen:
                    seen.add(v)
                    q.append(v)
        return False

    return all(a == b or reachable(a, b) for a, b in zip(sentence1, sentence2))


def answer_similar_iii(sentence1, sentence2):
    a, b = sentence1.split(), sentence2.split()
    if len(a) > len(b):
        a, b = b, a
    i = 0
    while i < len(a) and a[i] == b[i]:
        i += 1
    j = 0
    while j < len(a) - i and a[-1 - j] == b[-1 - j]:
        j += 1
    return i + j == len(a)


assert answer_build_pair_set([["a", "b"]]) == {("a", "b"), ("b", "a")}
assert answer_build_pair_set([]) == set()
assert answer_are_similar(["great", "acting", "skills"], ["fine", "drama", "talent"], P1) is True
assert answer_are_similar(["great"], ["doubleplus", "good"], [["great", "doubleplus"]]) is False
assert answer_ignore_case(["Great"], ["great"], []) is True
assert answer_ignore_case(["GREAT"], ["Fine"], [["great", "fine"]]) is True
assert answer_ignore_case(["a"], ["c"], [["A", "b"], ["b", "C"]]) is False
pairs = [["great", "good"], ["fine", "good"], ["drama", "acting"], ["skills", "talent"]]
assert answer_transitive_dfs(["great", "acting", "skills"], ["fine", "drama", "talent"], pairs) is True
assert answer_transitive_dfs(["great"], ["drama"], pairs) is False
assert answer_similar_iii("My name is Haley", "My Haley") is True
assert answer_similar_iii("of", "A lot of words") is False
assert answer_similar_iii("Eating right now", "Eating") is True
print("Tất cả đáp án PASS")

Bài 1: False True False
Tất cả đáp án PASS


In [30]:
rng = random.Random(737)
for _ in range(2000):
    vocab = list("abcdef")
    n = rng.randint(1, 4)
    s1 = [rng.choice(vocab) for _ in range(n)]
    s2 = [rng.choice(vocab) for _ in range(n)]
    pairs = [[rng.choice(vocab), rng.choice(vocab)] for _ in range(rng.randint(0, 5))]
    assert answer_transitive_dfs(s1, s2, pairs) == similar_transitive(s1, s2, pairs)
print("Bài 5 (DFS) khớp Union-Find trên 2000 bộ ngẫu nhiên")

Bài 5 (DFS) khớp Union-Find trên 2000 bộ ngẫu nhiên


Ghi chú bài 6: điều kiện `j < len(a) - i` ngăn phần hậu tố "đếm chồng" lên phần tiền tố đã khớp (ví dụ `a = ["x"]`, `b = ["x", "x"]`).

## 17. Template nhớ nhanh + Checklist submit

```python
class Solution:
    def areSentencesSimilar(self, sentence1, sentence2, similarPairs):
        if len(sentence1) != len(sentence2):          # zip không tự kiểm độ dài
            return False
        pair_set = set()
        for x, y in similarPairs:                     # đối xứng: thêm 2 chiều
            pair_set.add((x, y))
            pair_set.add((y, x))
        for w1, w2 in zip(sentence1, sentence2):
            if w1 != w2 and (w1, w2) not in pair_set: # phản xạ: w1 == w2 luôn ổn
                return False
        return True
```

**Câu thần chú:** *"độ dài trước — set tuple hai chiều — bằng nhau hoặc có cặp — không bắc cầu."*

**Checklist trước khi submit:**
- [ ] Kiểm `len(sentence1) != len(sentence2)` trước khi `zip`?
- [ ] Thêm **cả hai chiều** của mỗi cặp (hoặc dùng `frozenset`)?
- [ ] Từ giống hệt nhau luôn được chấp nhận (kể cả khi không có trong cặp)?
- [ ] Dùng `tuple` (không phải `list`) làm phần tử của set?
- [ ] **Không** suy luận bắc cầu (đó là bài 737)?
- [ ] Giữ phân biệt hoa thường như đề?